In [ ]:

"""
SHARED SETUP - Run ONCE at the beginning
"""

print("\n" + "="*80)
print("SETUP: DEPENDENCIES & DATA")
print("="*80)

# Install dependencies
!pip install -q transformers peft timm accelerate torch torchvision segmentation-models-pytorch scikit-learn pandas matplotlib seaborn

# Mount drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/harshini2806/lora_paper.git
%cd lora_paper

import os
os.makedirs('/content/drive/MyDrive/lora_experiments', exist_ok=True)

# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import ViTForImageClassification, ViTConfig
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score
import numpy as np
import time
from collections import Counter
import pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'



# Load data
from dataloader import GenDEBRIS

train_ds = GenDEBRIS(mode='train', path='/content/lora_paper/data')
val_ds = GenDEBRIS(mode='val', path='/content/lora_paper/data')

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

# Compute class weights
label_counts = Counter()
for _, mask in train_ds:
    valid = mask[mask != -1].flatten()
    label_counts.update(valid.tolist())

total = sum(label_counts.values())
freqs = np.array([label_counts.get(i, 0)/total for i in range(11)])
weights = 1.0 / (freqs + 1e-6)
weights = weights / weights.sum()
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

# Metrics function
def compute_metrics(preds, labels, num_classes=11):
    if isinstance(preds, torch.Tensor):
        preds = torch.argmax(preds, dim=1).cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    valid = labels != -1
    preds_v = preds[valid]
    labels_v = labels[valid]

    accuracy = (preds_v == labels_v).sum() / labels_v.size
    f1 = f1_score(labels_v, preds_v, average='weighted', zero_division=0)

    present_classes = np.unique(labels_v)
    ious = []
    for cls in present_classes:
        pred_m = (preds_v == cls)
        true_m = (labels_v == cls)
        inter = (pred_m & true_m).sum()
        union = (pred_m | true_m).sum()
        if union > 0:
            ious.append(inter / union)
    mean_iou = np.mean(ious) if len(ious) > 0 else 0.0

    return {'Accuracy': accuracy, 'F1': f1, 'mIoU': mean_iou}



# Training function
def train(model, name, epochs=70, lr=None):
    print(f"\n{'#'*70}\n# {name}\n{'#'*70}\n")

    if lr is None:
        lr = 2e-4 if 'lora' in name.lower() else 5e-5

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-1)
    scaler = GradScaler()

    train_dl = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
    val_dl = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2)

    best_metrics = {'mIoU':0, 'F1':0, 'Accuracy':0}
    t0 = time.time()

    for ep in range(epochs):
        model.train()
        total_loss = 0

        for imgs, masks in train_dl:
            if imgs.dim()==4 and imgs.shape[-1]==11:
                imgs = imgs.permute(0,3,1,2)
            imgs = F.interpolate(imgs, size=(224,224), mode='bilinear', align_corners=False)
            imgs, masks = imgs.float().to(device), masks.long().to(device)

            opt.zero_grad()
            with autocast("cuda"):
                out = model(imgs)
                if out.shape[2:] != masks.shape[1:]:
                    out = F.interpolate(out, masks.shape[1:], mode='bilinear')
                loss = loss_fn(out, masks)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total_loss += loss.item()

        scheduler.step()

        # Validation
        model.eval()
        val_metrics = {'mIoU':0, 'F1':0, 'Accuracy':0}
        with torch.no_grad():
            for imgs, masks in val_dl:
                if imgs.dim()==4 and imgs.shape[-1]==11:
                    imgs = imgs.permute(0,3,1,2)
                imgs = F.interpolate(imgs, size=(224,224), mode='bilinear', align_corners=False)
                imgs, masks = imgs.float().to(device), masks.long().to(device)

                with autocast("cuda"):
                    out = model(imgs)
                    if out.shape[2:] != masks.shape[1:]:
                        out = F.interpolate(out, masks.shape[1:], mode='bilinear')

                met = compute_metrics(out, masks)
                for k in val_metrics:
                    val_metrics[k] += met[k]

        for k in val_metrics:
            val_metrics[k] /= len(val_dl)

        print(f"Ep {ep+1:02d} | Loss={total_loss/len(train_dl):.4f} | mIoU={val_metrics['mIoU']:.4f} | "
              f"F1={val_metrics['F1']:.4f} | Acc={val_metrics['Accuracy']:.4f}")

        if val_metrics['mIoU'] > best_metrics['mIoU']:
            best_metrics = val_metrics.copy()
            torch.save(model.state_dict(), f"/content/drive/MyDrive/lora_experiments/{name}_best.pth")

    best_metrics['name'] = name
    best_metrics['time_min'] = (time.time()-t0)/60
    return best_metrics

print("✅ SETUP COMPLETE - Ready to train models!")



SETUP: DEPENDENCIES & DATA
Mounted at /content/drive
Cloning into 'lora_paper'...
remote: Enumerating objects: 4432, done.
remote: Total 4432 (delta 0), reused 0 (delta 0), pack-reused 4432 (from 1)
Receiving objects: 100% (4432/4432), 1.08 GiB | 17.51 MiB/s, done.
Resolving deltas: 100% (2817/2817), done.
Updating files: 100% (4478/4478), done.
/content/lora_paper


Load train set to memory:   0%|          | 0/694 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(
Load val set to memory: 100%|██████████| 328/328 [00:05<00:00, 55.69it/s]


Train: 694, Val: 328
✅ SETUP COMPLETE - Ready to train models!


In [ ]:
!pip install segmentation-models-pytorch==0.3.3
!pip install torch torchvision albumentations


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 52.7 MB/s eta 0:00:00
  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=2f4ae8b5af5fd9236f3d6e4b43aa6206b2f888e67df4f6d2f71732d923830e5a
  Stored in directory: /root/.cache/pip/wheels/9c/3f/43/e6271c7026fe08c185da2be23c98c8e87477d3db63f41f32ad
  Created wheel for pretrainedmodels: filename=pretrainedmodels-0.7.4-py3-none-any.whl size=60945 sha256=c49f6f3a1d03fe50d7b87ea4dacb7b8d24fb3eb85df7c396b7f5d91297543586
  Stored in directory: /root/.cache/pip/wheels/4c/01/56/40a48f75dbdfe167a0cb70d3b48913369a00ec5c4e9fed5f2b
Successfully built efficientnet-pytorch

In [ ]:
from segmentation_models_pytorch import Unet
import torch.nn as nn

def create_unet_baseline():
    print("🟡 Creating UNet Baseline (ResNet50 encoder)")

    model = Unet(
        encoder_name="resnet50",
        encoder_weights="imagenet",
        in_channels=11,           # dataset input channels
        classes=11,               # number of segmentation classes
    )
    return model


In [ ]:
from peft import LoraConfig, get_peft_model

def create_unet_lora(lora_r=16):
    model = create_unet_baseline()
    print(f"🔷 Adding LoRA adapters with r={lora_r}")

    lora_cfg = LoraConfig(
        r=lora_r,
        lora_alpha=lora_r*2,
        target_modules=[
            "conv1", "conv2", "conv3", "conv4", "downsample.0"
        ],
        lora_dropout=0.1,
        bias="none",
        modules_to_save=["decoder", "segmentation_head"]  # keeping decoder trainable
    )

    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model


In [ ]:
baseline_unet = create_unet_baseline().to(device)
result_baseline = train(baseline_unet, "unet_baseline_70ep", epochs=70, lr=5e-5)

lora_unet = create_unet_lora(lora_r=16).to(device)
result_lora = train(lora_unet, "unet_lora_r16_70ep", epochs=70, lr=5e-5)


🟡 Creating UNet Baseline (ResNet50 encoder)

######################################################################
# unet_baseline_70ep
######################################################################

Ep 01 | Loss=2.5343 | mIoU=0.0992 | F1=0.2355 | Acc=0.1968
Ep 02 | Loss=2.1449 | mIoU=0.1334 | F1=0.2625 | Acc=0.2483
Ep 03 | Loss=1.9420 | mIoU=0.1436 | F1=0.2973 | Acc=0.2703
Ep 04 | Loss=1.7280 | mIoU=0.1785 | F1=0.3406 | Acc=0.3070
Ep 05 | Loss=1.7446 | mIoU=0.2215 | F1=0.4083 | Acc=0.3768
Ep 06 | Loss=1.5037 | mIoU=0.2407 | F1=0.4608 | Acc=0.4430
Ep 07 | Loss=1.4467 | mIoU=0.2581 | F1=0.4739 | Acc=0.4313
Ep 08 | Loss=1.3558 | mIoU=0.2991 | F1=0.5231 | Acc=0.4904
Ep 09 | Loss=1.2585 | mIoU=0.3333 | F1=0.5610 | Acc=0.5244
Ep 10 | Loss=1.2392 | mIoU=0.3115 | F1=0.5529 | Acc=0.5188
Ep 11 | Loss=1.1141 | mIoU=0.3312 | F1=0.5590 | Acc=0.5244
Ep 12 | Loss=1.1006 | mIoU=0.3402 | F1=0.5828 | Acc=0.5399
Ep 13 | Loss=0.9439 | mIoU=0.3360 | F1=0.6005 | Acc=0.5640
Ep 14 | Loss=0.8313 | mI